# RMIA: Relative Membership Inference Attack

**EECE 608 -- Mohamad Faour**

Uses production RMIA code from ML Privacy Meter (Zarifzadeh et al., ICML 2024)
to test whether a stronger attack closes more of the tightness gap.

Compares Raw LiRA vs RMIA across the sigma sweep.

## 0. Setup

In [ ]:
import os, sys, shutil
from pathlib import Path

WORK = Path("/content/EECE_608")
if WORK.exists():
    shutil.rmtree(WORK)
!git clone https://github.com/mohdfaour03/EECE_608.git /content/EECE_608

os.chdir(WORK)
!pip install -q opacus>=1.5 dp-accounting>=0.4 pyyaml scikit-learn
!pip install -e . -q

sys.path.insert(0, str(WORK / "src"))
sys.path.insert(0, str(WORK / "autoresearch"))

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("Ready.")

## 1. Imports and helpers

In [ ]:
import prepare
import torch
import torch.nn.functional as F
import random
import time
import logging
import statistics
import numpy as np
from pathlib import Path
from torch.utils.data import Subset

from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.data.datasets import load_dataset_bundle, DatasetBundle
from dp_audit_tightness.utils.seeds import set_global_seed
from dp_audit_tightness.models.io import load_model_for_inference

# Production RMIA code from ML Privacy Meter
from rmia_attacks import run_rmia

logging.basicConfig(level=logging.WARNING, format='%(message)s')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bundle = load_dataset_bundle(
    prepare._build_train_config().dataset,
    split_seed=prepare.SPLIT_SEED,
)
n_train = len(bundle.train_dataset)
n_eval = len(bundle.eval_dataset)
print(f"Dataset: {n_train} train, {n_eval} eval")
print(f"Device: {device}")

In [ ]:
def train_model(noise_multiplier, seed, dataset=None):
    """Train a DP-SGD model with given noise_multiplier. Returns (model, record)."""
    config = prepare._build_train_config(
        noise_multiplier=noise_multiplier, seed=seed
    )
    set_global_seed(seed)
    logger = logging.getLogger(f"train_s{seed}_nm{noise_multiplier}")
    logger.setLevel(logging.WARNING)

    if dataset is not None:
        ds_bundle = DatasetBundle(
            train_dataset=dataset,
            eval_dataset=bundle.eval_dataset,
            input_dim=prepare.INPUT_DIM,
            num_classes=prepare.NUM_CLASSES,
            train_size=len(dataset),
            eval_size=len(bundle.eval_dataset),
        )
        outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=ds_bundle)
    else:
        outcome = train_dp_sgd(config=config, logger=logger)

    model = load_model_for_inference(config.model, outcome.checkpoint_path, device=device)
    return model, outcome.record


def get_softmax_signals(model, dataset, indices, batch_size=512):
    """Get softmax probability of the true class for each example. Shape: (len(indices), 1)."""
    all_probs = []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            bi = indices[s:s+batch_size]
            imgs, lbls = zip(*(dataset[i] for i in bi))
            x = torch.stack(imgs).to(device)
            y = torch.tensor(lbls, dtype=torch.long, device=device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            true_class_probs = probs.gather(1, y.unsqueeze(1))
            all_probs.append(true_class_probs.cpu().numpy())
    return np.concatenate(all_probs)  # shape (n, 1)


print("Helpers defined.")

## 2. RMIA + LiRA comparison across sigma

For each sigma:
1. Train target model + K shadow models
2. Collect softmax signals for all models on query + population points
3. Run RMIA (production code) and Raw LiRA
4. Compare tightness

The production `run_rmia` expects:
- `all_signals[i, j]` = softmax prob of example i under model j
- `population_signals[i, j]` = softmax prob of population example i under model j
- `all_memberships[i, j]` = True if example i was in training set of model j
- target_model_idx = which column is the target model

In [ ]:
SIGMAS = [0.5, 0.8, 1.1, 1.5, 2.0, 3.0, 5.0]
K = 32  # shadow models
BUDGET = 256
AUDIT_SEEDS = [401, 402, 403, 404, 405]
N_POPULATION = 500  # reference population size for RMIA

all_results = []

print("=" * 110)
print(f"RMIA vs LiRA SWEEP: K={K}, budget={BUDGET}, {len(AUDIT_SEEDS)} seeds, pop={N_POPULATION}")
print("=" * 110)

for sigma in SIGMAS:
    t0 = time.time()
    print(f"\n{'='*80}")
    print(f"SIGMA = {sigma}")
    print(f"{'='*80}")

    # --- Train target model ---
    target_model, record = train_model(sigma, seed=prepare.TRAINING_SEED)
    eps_rdp = record.epsilon_upper_theory
    eps_pld = record.epsilon_upper_pld
    if eps_pld is None:
        try:
            sampling_rate = prepare.BATCH_SIZE / n_train
            num_steps = prepare.EPOCHS * (n_train // prepare.BATCH_SIZE)
            pld_result = compute_epsilon_pld(
                noise_multiplier=sigma, sampling_rate=sampling_rate,
                num_steps=num_steps, delta=prepare.DELTA)
            eps_pld = pld_result["epsilon_pld"]
        except Exception:
            eps_pld = eps_rdp
    accuracy = record.utility_metrics.get("accuracy", 0)
    print(f"  Target: eps_rdp={eps_rdp:.4f} eps_pld={eps_pld:.4f} acc={accuracy:.4f}")

    # --- Train shadow models ---
    half = n_train // 2
    shadow_models = []
    shadow_member_sets = []
    print(f"  Training {K} shadow models...")
    for k in range(K):
        shadow_seed = 5000 + k
        rng_k = random.Random(shadow_seed)
        all_indices = list(range(n_train))
        rng_k.shuffle(all_indices)
        in_indices = sorted(all_indices[:half])
        shadow_member_sets.append(set(in_indices))
        subset = Subset(bundle.train_dataset, in_indices)
        sm, _ = train_model(sigma, shadow_seed, dataset=subset)
        shadow_models.append(sm)
    print(f"  Shadow models trained in {time.time()-t0:.0f}s")

    # --- Collect query indices ---
    all_member_idx, all_nonmember_idx = [], []
    for seed in AUDIT_SEEDS:
        rng = random.Random(seed)
        all_member_idx.extend(rng.sample(range(n_train), BUDGET // 2))
        all_nonmember_idx.extend(rng.sample(range(n_eval), BUDGET // 2))

    # Population z-points: random eval examples NOT in query set
    pop_rng = random.Random(999)
    nonmember_set = set(all_nonmember_idx)
    pop_candidates = [i for i in range(n_eval) if i not in nonmember_set]
    pop_indices = pop_rng.sample(pop_candidates, min(N_POPULATION, len(pop_candidates)))

    # Combined query: members then non-members
    n_mem = len(all_member_idx)
    n_non = len(all_nonmember_idx)
    n_query = n_mem + n_non

    # --- Build signal matrices for RMIA ---
    # all_signals shape: (n_query, 1 + K) -- column 0 is target, columns 1..K are shadows
    # We treat target model as index 0, shadows as index 1..K
    # Need paired model at index 1 for tune_offline_a, so we use shadow 0 as paired
    n_models = 1 + K  # target + shadows
    target_model_idx = 0

    print(f"  Collecting signals ({n_query} query + {len(pop_indices)} population)...")

    all_signals = np.zeros((n_query, n_models), dtype=np.float32)
    population_signals = np.zeros((len(pop_indices), n_models), dtype=np.float32)
    all_memberships = np.zeros((n_query, n_models), dtype=bool)

    # Target model signals
    mem_sigs = get_softmax_signals(target_model, bundle.train_dataset, all_member_idx)
    non_sigs = get_softmax_signals(target_model, bundle.eval_dataset, all_nonmember_idx)
    all_signals[:n_mem, 0] = mem_sigs.ravel()
    all_signals[n_mem:, 0] = non_sigs.ravel()
    pop_sigs = get_softmax_signals(target_model, bundle.eval_dataset, pop_indices)
    population_signals[:, 0] = pop_sigs.ravel()

    # Membership for target model: members are True, non-members are False
    all_memberships[:n_mem, 0] = True
    all_memberships[n_mem:, 0] = False

    # Shadow model signals
    for k, sm in enumerate(shadow_models):
        col = k + 1
        sm_mem = get_softmax_signals(sm, bundle.train_dataset, all_member_idx)
        sm_non = get_softmax_signals(sm, bundle.eval_dataset, all_nonmember_idx)
        all_signals[:n_mem, col] = sm_mem.ravel()
        all_signals[n_mem:, col] = sm_non.ravel()

        sm_pop = get_softmax_signals(sm, bundle.eval_dataset, pop_indices)
        population_signals[:, col] = sm_pop.ravel()

        # Shadow membership: for member queries, check if index was in shadow's training set
        for i, idx in enumerate(all_member_idx):
            all_memberships[i, col] = idx in shadow_member_sets[k]
        # Non-members are never in any shadow's training set (they're eval data)
        all_memberships[n_mem:, col] = False

    print(f"  Signals collected. Running attacks...")

    # --- Run RMIA (production code) ---
    rmia_scores = run_rmia(
        target_model_idx=target_model_idx,
        all_signals=all_signals,
        population_signals=population_signals,
        all_memberships=all_memberships,
        num_reference_models=min(K // 2, 16),  # use up to 16 reference models
        offline_a=0.3,  # default from ML Privacy Meter
    )
    rmia_member_scores = rmia_scores[:n_mem].tolist()
    rmia_nonmember_scores = rmia_scores[n_mem:].tolist()

    # --- Run Raw LiRA for comparison ---
    lira_m, lira_n = [], []
    # Precompute loss matrices for LiRA
    for i, idx in enumerate(all_member_idx):
        in_losses, out_losses = [], []
        for k in range(K):
            # Use -log(softmax) as loss proxy
            sig = all_signals[i, k + 1]
            loss_k = -np.log(max(sig, 1e-10))
            if idx in shadow_member_sets[k]:
                in_losses.append(loss_k)
            else:
                out_losses.append(loss_k)
        if in_losses and out_losses:
            lira_m.append(np.mean(out_losses) - np.mean(in_losses))
        else:
            lira_m.append(0.0)

    # Non-member LiRA: shadow_mean_loss - target_loss
    for i, idx in enumerate(all_nonmember_idx):
        target_sig = all_signals[n_mem + i, 0]
        target_loss = -np.log(max(target_sig, 1e-10))
        shadow_losses = [-np.log(max(all_signals[n_mem + i, k + 1], 1e-10)) for k in range(K)]
        lira_n.append(np.mean(shadow_losses) - target_loss)

    # --- Evaluate both ---
    for attack_name, m_scores, n_scores in [
        ("RMIA", rmia_member_scores, rmia_nonmember_scores),
        ("Raw LiRA", lira_m, lira_n),
    ]:
        est_cons = estimate_empirical_lower_bound(
            member_scores=m_scores, nonmember_scores=n_scores,
            delta=prepare.DELTA, align_event_to_score_direction=True,
            require_member_favoring=True, report_confidence_supported_lower_bound=True)
        est_point = estimate_empirical_lower_bound(
            member_scores=m_scores, nonmember_scores=n_scores,
            delta=prepare.DELTA, align_event_to_score_direction=True,
            require_member_favoring=True, report_confidence_supported_lower_bound=False)

        eps_lc = est_cons.epsilon_lower_empirical
        eps_lp = est_point.epsilon_lower_empirical
        tr_rdp = eps_lc / eps_rdp if eps_rdp > 0 else 0
        tr_pld = eps_lc / eps_pld if eps_pld > 0 else 0
        gap = statistics.fmean(m_scores) - statistics.fmean(n_scores)

        row = {
            "sigma": sigma, "attack": attack_name,
            "eps_rdp": eps_rdp, "eps_pld": eps_pld,
            "eps_lower_cons": eps_lc, "eps_lower_point": eps_lp,
            "tr_cons_rdp": tr_rdp, "tr_cons_pld": tr_pld,
            "score_gap": gap, "accuracy": accuracy,
        }
        all_results.append(row)
        print(f"  {attack_name:10s}: eps_lower cons={eps_lc:.4f} point={eps_lp:.4f}  "
              f"tr(RDP)={tr_rdp:.4%} tr(PLD)={tr_pld:.4%}  gap={gap:.4f}")

    print(f"  Time: {time.time()-t0:.0f}s")

print("\n" + "=" * 110)
print("Sweep complete.")

## 3. Results

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

print("=" * 120)
print("RMIA vs LiRA RESULTS")
print("=" * 120)
print(df[["sigma", "attack", "eps_rdp", "eps_pld",
          "eps_lower_cons", "eps_lower_point",
          "tr_cons_rdp", "tr_cons_pld", "score_gap"]].to_string(index=False))

print()
print("-" * 80)
print("HEAD-TO-HEAD: RMIA vs LiRA conservative tightness (vs PLD):")
for sigma in SIGMAS:
    rmia_row = df[(df["sigma"] == sigma) & (df["attack"] == "RMIA")]
    lira_row = df[(df["sigma"] == sigma) & (df["attack"] == "Raw LiRA")]
    if len(rmia_row) > 0 and len(lira_row) > 0:
        r = rmia_row.iloc[0]
        l = lira_row.iloc[0]
        winner = "RMIA" if r["tr_cons_pld"] > l["tr_cons_pld"] else "LiRA"
        print(f"  sigma={sigma:.1f}: RMIA={r['tr_cons_pld']:.4%}  LiRA={l['tr_cons_pld']:.4%}  winner={winner}")

out_dir = WORK / "autoresearch" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "rmia_vs_lira_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved to {csv_path}")

## 4. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rmia_df = df[df["attack"] == "RMIA"]
lira_df = df[df["attack"] == "Raw LiRA"]

# Plot 1: Conservative tightness vs RDP
ax = axes[0]
ax.plot(rmia_df["sigma"], rmia_df["tr_cons_rdp"] * 100, "o-", label="RMIA", color="red")
ax.plot(lira_df["sigma"], lira_df["tr_cons_rdp"] * 100, "s-", label="Raw LiRA", color="blue")
ax.set_xlabel("Noise Multiplier (sigma)")
ax.set_ylabel("Conservative Tightness vs RDP (%)")
ax.set_title("RMIA vs LiRA: Tightness vs RDP")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Conservative tightness vs PLD
ax = axes[1]
ax.plot(rmia_df["sigma"], rmia_df["tr_cons_pld"] * 100, "o-", label="RMIA", color="red")
ax.plot(lira_df["sigma"], lira_df["tr_cons_pld"] * 100, "s-", label="Raw LiRA", color="blue")
ax.set_xlabel("Noise Multiplier (sigma)")
ax.set_ylabel("Conservative Tightness vs PLD (%)")
ax.set_title("RMIA vs LiRA: Tightness vs PLD")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = WORK / "autoresearch" / "results" / "rmia_vs_lira_plots.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {fig_path}")

## 5. Download

In [ ]:
from google.colab import files
files.download(str(csv_path))
files.download(str(fig_path))
print("Downloads triggered.")